## Transform Addresses Data
1. Create one record for each customer with 2 sets of address columns, 1 for shipping and 1 for billing address 
1. Write transformed data to the Silver schema  

In [0]:
df_addresses = spark.table('gizmobox.bronze.py_addresses')
display(df_addresses)

customer_id,address_type,address_line_1,city,state,postcode
9179,shipping,9713 Medina Mountains Apt. 813,Lake Erika,Mississippi,91945
9179,billing,084 Anne Hollow Apt. 064,East Jasontown,Minnesota,77329
5816,shipping,8400 Rebecca Lodge Suite 011,Lake Lisatown,Montana,11354
5816,billing,869 Linda Locks Apt. 105,West Melissabury,Michigan,37662
4858,shipping,4838 Jacob Neck,Jamesview,Wisconsin,58657
4858,billing,4558 Cabrera Islands,Lake David,Rhode Island,06133
7207,shipping,75470 Richardson Mission Apt. 147,New Rachelhaven,Hawaii,38857
7207,billing,4450 Michelle Mission Suite 663,North Justin,Kentucky,91757
8539,shipping,331 Monica Route Apt. 022,Michellechester,Texas,14637
8539,billing,48794 Aguirre Ferry Apt. 952,North Amy,Washington,92507


### 1. Create one record for each customer with both addresses, one for each address_type

In [0]:
from pyspark.sql import functions as F
df_pivoted_addresses = (
    df_addresses
    .groupBy('customer_id')
    .pivot('address_type', ['billing', 'shipping'])
    .agg(
        F.max('address_line_1').alias('address_line_1'),
        F.max('city').alias('city'),
        F.max('state').alias('state'),
        F.max('postcode').alias('postcode')
    )
)
display(df_pivoted_addresses)

customer_id,billing_address_line_1,billing_city,billing_state,billing_postcode,shipping_address_line_1,shipping_city,shipping_state,shipping_postcode
1211,7938 Cervantes Mountain Suite 159,Oconnorhaven,Indiana,44129,580 Samantha Lodge,New Christinaborough,Louisiana,04251
1987,3440 Beck Parkways,West Jennifer,Alaska,79766,944 Smith Squares,Port Jennifer,Maryland,28646
2054,890 Brandon Brook Suite 774,North Nicoleborough,Wyoming,15077,225 Lopez Bridge Suite 386,South Monica,Indiana,34134
2141,7451 Lisa Stravenue Suite 994,South Victor,Kansas,49835,098 Sellers Station,South Javierstad,Texas,59689
2344,5008 Corey Landing Apt. 475,South Christinaburgh,New Mexico,03650,97619 Sanchez Prairie Suite 359,Kelleyfurt,Louisiana,64906
2639,222 Dominique Springs Apt. 942,Colechester,Louisiana,96258,5383 Albert Grove Apt. 804,North Destinyland,Illinois,60150
2703,7639 Derek Streets,New Mary,California,68610,5385 Paul Port Suite 963,Richardsonton,Florida,06432
3084,78828 Montgomery Branch Suite 316,South Chelsea,Wisconsin,61054,3337 Gregory Island,Mitchellbury,Tennessee,70603
3295,36789 Hatfield Landing,Haydenton,Tennessee,60950,9569 Dennis Way Suite 047,North Amy,New York,17867
3409,1294 Edward Inlet,North Alison,Indiana,82752,918 April Junctions,New Theresa,Indiana,46004


### 2. Write transformed data to the Silver schema 

In [0]:
df_pivoted_addresses.writeTo('gizmobox.silver.py_addresses').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_addresses ;

customer_id,billing_address_line_1,billing_city,billing_state,billing_postcode,shipping_address_line_1,shipping_city,shipping_state,shipping_postcode
1211,7938 Cervantes Mountain Suite 159,Oconnorhaven,Indiana,44129,580 Samantha Lodge,New Christinaborough,Louisiana,04251
1987,3440 Beck Parkways,West Jennifer,Alaska,79766,944 Smith Squares,Port Jennifer,Maryland,28646
2054,890 Brandon Brook Suite 774,North Nicoleborough,Wyoming,15077,225 Lopez Bridge Suite 386,South Monica,Indiana,34134
2141,7451 Lisa Stravenue Suite 994,South Victor,Kansas,49835,098 Sellers Station,South Javierstad,Texas,59689
2344,5008 Corey Landing Apt. 475,South Christinaburgh,New Mexico,03650,97619 Sanchez Prairie Suite 359,Kelleyfurt,Louisiana,64906
2639,222 Dominique Springs Apt. 942,Colechester,Louisiana,96258,5383 Albert Grove Apt. 804,North Destinyland,Illinois,60150
2703,7639 Derek Streets,New Mary,California,68610,5385 Paul Port Suite 963,Richardsonton,Florida,06432
3084,78828 Montgomery Branch Suite 316,South Chelsea,Wisconsin,61054,3337 Gregory Island,Mitchellbury,Tennessee,70603
3295,36789 Hatfield Landing,Haydenton,Tennessee,60950,9569 Dennis Way Suite 047,North Amy,New York,17867
3409,1294 Edward Inlet,North Alison,Indiana,82752,918 April Junctions,New Theresa,Indiana,46004
